# Gene expression by group

Aggregate expression of a gene across one (or two) categorical obs columns and visualise with:
- **Boxplot** — full distribution per group
- **Barplot with 95% CI** — mean ± confidence interval

In [ ]:
try:
    import micropip
    # 'vark' is vendored next to the notebook (top-level package).
    await micropip.install(['pandas', 'matplotlib', 'seaborn'])
except:
    pass  # already installed in standard Python
print('ready')

In [ ]:
from vark import AnthiveClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import js
    api_url = f"{js.location.origin}/api/"
except:
    api_url = None

client = AnthiveClient(api_url)
print(f'Connected to {client.base_url}')

## Configuration

In [ ]:
db_id    = 'SF9/SAN'    # dataset to query
gene     = 'Apoe'       # gene of interest
group1   = 'genotype'   # primary grouping (x-axis)
group2   = None         # optional second grouping (hue); set None to disable

info = client.get_database_info(db_id)
layer = info.get('default_layer')
if layer is None:
    raise ValueError(f"{db_id} has no default_layer — pick a different dataset or set it with ant-duckdb-check")

print(f'Database : {db_id}')
print(f'Layer    : {layer}')
print(f'Obs cats : {info.get("obs", {}).get("categorical", [])}')

## Load data

In [ ]:
metadata_cols = [c for c in [group1, group2] if c]
data = client.get_cells(db_id, genes=[gene], metadata=metadata_cols, layer=layer, format='dataframe')
data = data.dropna(subset=[group1])

if group2:
    data = data.dropna(subset=[group2])

print(f'Cells: {len(data):,}')
print(f'{group1} values: {sorted(data[group1].unique())}')
if group2:
    print(f'{group2} values: {sorted(data[group2].unique())}')
data.head(3)

## Boxplot

Shows the full distribution. Outliers are hidden for clarity. Dots are a random sample of up to 2,000 cells.

In [ ]:
n_groups = data[group1].nunique()
fig, ax = plt.subplots(figsize=(max(5, n_groups * 0.9), 5))

sns.boxplot(data=data, x=group1, y=gene, hue=group2 if group2 else None,
            ax=ax, showfliers=False, palette='tab10')

# Overlay a random sample of points
sample = data.sample(min(2000, len(data)), random_state=0)
sns.stripplot(data=sample, x=group1, y=gene, hue=group2 if group2 else None,
              ax=ax, size=2, alpha=0.35, color='k',
              dodge=bool(group2), legend=False)

ax.set_xlabel(group1)
ax.set_ylabel(f'{gene} expression ({layer})')
ax.set_title(f'{gene} by {group1}' + (f' × {group2}' if group2 else '') + f'  [{db_id}]')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

## Barplot with 95% confidence interval

CI = 1.96 × SE = 1.96 × (std / √n). Bars are sorted by mean.

In [ ]:
group_cols = [g for g in [group1, group2] if g]
agg = (
    data.groupby(group_cols)[gene]
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
agg['ci95'] = 1.96 * agg['std'] / np.sqrt(agg['count'])

if group2:
    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(max(6, agg[group1].nunique() * 1.2), 5))
    hue_vals = sorted(agg[group2].unique())
    x = np.arange(agg[group1].nunique())
    width = 0.8 / len(hue_vals)
    palette = plt.cm.tab10.colors
    for i, hval in enumerate(hue_vals):
        sub = agg[agg[group2] == hval].set_index(group1).reindex(sorted(agg[group1].unique()))
        ax.bar(x + i * width, sub['mean'], width,
               yerr=sub['ci95'], capsize=3,
               label=str(hval), color=palette[i % len(palette)], alpha=0.8)
    ax.set_xticks(x + width * (len(hue_vals) - 1) / 2)
    ax.set_xticklabels(sorted(agg[group1].unique()), rotation=40, ha='right')
    ax.legend(title=group2)
else:
    agg_s = agg.sort_values('mean', ascending=False)
    fig, ax = plt.subplots(figsize=(max(5, len(agg_s) * 0.9), 5))
    ax.bar(agg_s[group1], agg_s['mean'], yerr=agg_s['ci95'],
           capsize=4, color='steelblue', alpha=0.8, ecolor='gray')
    plt.xticks(rotation=40, ha='right')

ax.set_ylabel(f'{gene} mean ± 95% CI ({layer})')
ax.set_title(f'{gene} by {group1}' + (f' × {group2}' if group2 else '') + f'  [{db_id}]')
plt.tight_layout()
plt.show()